In [ ]:
import numpy as np
import pandas as pd

#loads data in
import gendata as gen

# Import Splits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# Import Model
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV

# Import Metrics
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.special import huber
from sklearn.metrics import make_scorer, mean_pinball_loss # For Quantile Loss


In [ ]:
# Read in the data
df = gen.load_training_data()

In [ ]:
# Create a list of all categorical features (needed for CatBoost)
cat_features = gen.cat_features

# Clean Categorical Features to make sure everything is a string and no NaN values
df[cat_features] = (
    df[cat_features]
    .fillna("Missing")
    .astype(str)
)

# Create a list of all features and target
features = cat_features.copy()
features.extend(['PAT2PROV_DISTANCE','LENGTH_OF_STAY','NUM_CODES'])

['SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'PAT_AGE',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL']

In [75]:
# Extract a set with stratification to do hyperparameter tuning
X_hyp_train, X_hyp_test, y_hyp_train, y_hyp_test = train_test_split(df[features],df.LENGTH_OF_STAY,
                                                    shuffle = True,
                                                    random_state = 1990,
                                                    stratify = df['STRATA'],
                                                   test_size = .975)

In [57]:
# Verifying the size of the tuning train set
X_hyp_train.shape

(140131, 33)

In [ ]:
# Find the percentile associated with a 'prolonged' stay of 30 days. This will be the quantile used in Quantile.
quantile = df.loc[df.LENGTH_OF_STAY <= 30].shape[0]/df.shape[0]
quantile

In [60]:
#Hyperparameter Tuning with RMSE
param_grid = {
    "depth": [6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 10],
    "bagging_temperature": [0, 1, 3],
}

# strata_train = df.loc[X_hyp_train.index, 'STRATA']

# skf_hyp = StratifiedKFold(
#     n_splits=3,
#     shuffle=True,
#     random_state=2222
# )

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# # strata_train.value_counts()
# for fold, (train_idx, valid_idx) in enumerate(skf_hyp.split(X_hyp_train, strata_train)):
#     X_tr = X_hyp_train.iloc[train_idx].copy()
#     X_val = X_hyp_train.iloc[valid_idx].copy()
#     y_tr = y_hyp_train.iloc[train_idx]
#     y_val = y_hyp_train.iloc[valid_idx]
    
    # Instantiate Model
model = CatBoostRegressor(
        iterations=1000,
        loss_function = "RMSE",
        eval_metric = "RMSE",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=50
    )

grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = "neg_root_mean_squared_error",
        cv = 3,  
        verbose = 2,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, cat_features = cat_features)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Learning rate set to 0.034325
0:	learn: 11.7934185	total: 143ms	remaining: 7m 10s
100:	learn: 2.6160673	total: 7.69s	remaining: 3m 40s
200:	learn: 1.0747591	total: 14.9s	remaining: 3m 27s
300:	learn: 0.5038397	total: 21.5s	remaining: 3m 13s
400:	learn: 0.2678899	total: 28.1s	remaining: 3m 2s
500:	learn: 0.1832560	total: 34.3s	remaining: 2m 50s
600:	learn: 0.1463310	total: 40.1s	remaining: 2m 40s
700:	learn: 0.1173088	total: 46.6s	remaining: 2m 32s
800:	learn: 0.0967776	total: 53.5s	remaining: 2m 26s
900:	learn: 0.0823219	total: 1m	remaining: 2m 20s
1000:	learn: 0.0698711	total: 1m 6s	remaining: 2m 13s
1100:	learn: 0.0584355	total: 1m 13s	remaining: 2m 6s
1200:	learn: 0.0503669	total: 1m 19s	remaining: 1m 59s
1300:	learn: 0.0434811	total: 1m 26s	remaining: 1m 53s
1400:	learn: 0.0382061	total: 1m 33s	remaining: 1m 47s
1500:	learn: 0.0341213	total: 1m 40s	remaining: 1m 40s
1600:	learn: 0.0307088	total: 1m 47s	remaining: 1m 33s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...2, verbose=50)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bagging_temperature': [0, 1, ...], 'depth': [6, 7, ...], 'l2_leaf_reg': [3, 5, ...], 'learning_rate': [0.01, 0.03, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time f

In [61]:
#Hyperparameter Tuning with MAE
param_grid = {
    "depth": [6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 10],
    "bagging_temperature": [0, 1, 3],
}

# strata_train = df.loc[X_hyp_train.index, 'STRATA']

# skf_hyp = StratifiedKFold(
#     n_splits=3,
#     shuffle=True,
#     random_state=2222
# )

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# # strata_train.value_counts()
# for fold, (train_idx, valid_idx) in enumerate(skf_hyp.split(X_hyp_train, strata_train)):
#     X_tr = X_hyp_train.iloc[train_idx].copy()
#     X_val = X_hyp_train.iloc[valid_idx].copy()
#     y_tr = y_hyp_train.iloc[train_idx]
#     y_val = y_hyp_train.iloc[valid_idx]
    
    # Instantiate Model
model = CatBoostRegressor(
        iterations=1000,
        loss_function = "MAE",
        eval_metric = "MAE",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=50
    )

grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = "neg_mean_absolute_error",
        cv = 3,  
        verbose = 1,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, cat_features = cat_features)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
0:	learn: 3.2495265	total: 168ms	remaining: 2m 47s
50:	learn: 1.4801697	total: 4.95s	remaining: 1m 32s
100:	learn: 0.8103850	total: 9.26s	remaining: 1m 22s
150:	learn: 0.6121120	total: 13.6s	remaining: 1m 16s
200:	learn: 0.3949552	total: 18.2s	remaining: 1m 12s
250:	learn: 0.2768227	total: 22.6s	remaining: 1m 7s
300:	learn: 0.2278169	total: 26.6s	remaining: 1m 1s
350:	learn: 0.2098950	total: 30.4s	remaining: 56.3s
400:	learn: 0.1886581	total: 34.4s	remaining: 51.4s
450:	learn: 0.1789971	total: 38s	remaining: 46.3s
500:	learn: 0.1744777	total: 41.8s	remaining: 41.6s
550:	learn: 0.1599265	total: 44.9s	remaining: 36.6s
600:	learn: 0.1507436	total: 48.2s	remaining: 32s
650:	learn: 0.1462335	total: 51.8s	remaining: 27.7s
700:	learn: 0.1376803	total: 55s	remaining: 23.5s
750:	learn: 0.1352225	total: 58.4s	remaining: 19.3s
800:	learn: 0.1342637	total: 1m 1s	remaining: 15.3s
850:	learn: 0.1271263	total: 1m 5s	remaining: 11.4s
900:	l

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...2, verbose=50)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bagging_temperature': [0, 1, ...], 'depth': [6, 7, ...], 'l2_leaf_reg': [3, 5, ...], 'learning_rate': [0.01, 0.03, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for e

In [81]:
#Hyperparameter Tuning with Quantile (98.75)

param_grid = {
    "depth": [6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 10],
    "bagging_temperature": [0, 1, 3],
}
    
    # Instantiate Model
model = CatBoostRegressor(
        iterations=1000,
        loss_function = "Quantile:alpha=0.987531903035467",
        eval_metric = "Quantile:alpha=0.987531903035467",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=50
    )

grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring=make_scorer(
            mean_pinball_loss,
            alpha=0.987531903035467,
            greater_is_better=False
            ),
        cv = 3,  
        verbose = 1,
        n_jobs = 1
    )

grid_search.fit(X_hyp_train, y_hyp_train, cat_features = cat_features)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
0:	learn: 0.6642834	total: 58.3ms	remaining: 58.2s
50:	learn: 0.3160470	total: 923ms	remaining: 17.2s
100:	learn: 0.2536075	total: 1.63s	remaining: 14.5s
150:	learn: 0.2185568	total: 2.35s	remaining: 13.2s
200:	learn: 0.1922979	total: 3.1s	remaining: 12.3s
250:	learn: 0.1698545	total: 3.9s	remaining: 11.6s
300:	learn: 0.1497435	total: 4.71s	remaining: 10.9s
350:	learn: 0.1328436	total: 5.57s	remaining: 10.3s
400:	learn: 0.1171272	total: 6.31s	remaining: 9.43s
450:	learn: 0.1038977	total: 7.08s	remaining: 8.62s
500:	learn: 0.0930663	total: 7.83s	remaining: 7.79s
550:	learn: 0.0834722	total: 8.53s	remaining: 6.95s
600:	learn: 0.0747208	total: 9.24s	remaining: 6.14s
650:	learn: 0.0658346	total: 10s	remaining: 5.38s
700:	learn: 0.0585961	total: 10.8s	remaining: 4.62s
750:	learn: 0.0522822	total: 11.6s	remaining: 3.84s
800:	learn: 0.0471920	total: 12.3s	remaining: 3.06s
850:	learn: 0.0428195	total: 13.1s	remaining: 2.29s
900:	lea

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...2, verbose=50)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bagging_temperature': [0, 1, ...], 'depth': [6, 7, ...], 'l2_leaf_reg': [3, 5, ...], 'learning_rate': [0.01, 0.03, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",make_scorer(m...7531903035467)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time f

In [82]:
#Hyperparameter Tuning with Huber Loss

param_grid = {
    "depth": [6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 10],
    "bagging_temperature": [0, 1, 3],
}

def huber_loss(y_true, y_pred, delta=2.0):
    error = y_true - y_pred
    is_small = np.abs(error) <= delta

    squared = 0.5 * error**2
    linear = delta * (np.abs(error) - 0.5 * delta)

    return np.mean(np.where(is_small, squared, linear))

huber_scorer = make_scorer(
    huber_loss,
    delta=2.0,
    greater_is_better=False)
    
    # Instantiate Model
model = CatBoostRegressor(
        iterations=1000,
        loss_function = "Huber:delta=2.0",
        eval_metric = "Huber:delta=2.0",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=50
    )

grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = huber_scorer,
        cv = 3,  
        verbose = 1,
        n_jobs = 1
    )

grid_search.fit(X_hyp_train, y_hyp_train, cat_features = cat_features)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
0:	learn: 106.2247571	total: 28.3ms	remaining: 28.2s
50:	learn: 7.1768460	total: 629ms	remaining: 11.7s
100:	learn: 4.8488855	total: 1.17s	remaining: 10.4s
150:	learn: 3.0215898	total: 1.77s	remaining: 9.95s
200:	learn: 2.8132315	total: 2.41s	remaining: 9.57s
250:	learn: 2.5723483	total: 3.02s	remaining: 9.02s
300:	learn: 2.2992560	total: 3.62s	remaining: 8.41s
350:	learn: 2.1510225	total: 4.24s	remaining: 7.85s
400:	learn: 2.0452170	total: 4.88s	remaining: 7.29s
450:	learn: 1.9672776	total: 5.51s	remaining: 6.71s
500:	learn: 1.9096452	total: 6.16s	remaining: 6.14s
550:	learn: 1.8650592	total: 6.9s	remaining: 5.62s
600:	learn: 1.8192119	total: 7.57s	remaining: 5.03s
650:	learn: 1.7767164	total: 8.25s	remaining: 4.42s
700:	learn: 1.7381993	total: 8.94s	remaining: 3.81s
750:	learn: 1.6914558	total: 9.59s	remaining: 3.18s
800:	learn: 1.6628859	total: 10.2s	remaining: 2.54s
850:	learn: 1.6395150	total: 10.8s	remaining: 1.9s
900:

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...2, verbose=50)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bagging_temperature': [0, 1, ...], 'depth': [6, 7, ...], 'l2_leaf_reg': [3, 5, ...], 'learning_rate': [0.01, 0.03, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(h...t', delta=2.0)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time

<u>**RMSE Hyperparameter Best Estimator**<u>

CatBoostRegressor(**bagging_temperature=0,\
                depth=6**, \
                early_stopping_rounds=100, \
                eval_metric='RMSE', \
                iterations=1000, \
                **l2_leaf_reg=3,\
                learning_rate=0.03**,\
                loss_function='RMSE',\
                verbose=50)

<u>**MAE Hyperparameter Best Estimator**<u>

CatBoostRegressor(**bagging_temperature=0,\
                depth=7**, \
                early_stopping_rounds=100, \
                eval_metric='MAE', \
                iterations=1000, \
                **l2_leaf_reg=5, \
                learning_rate=0.05**, \
                loss_function='MAE', \
                verbose=50)

<u>**Quantile Hyperparameter Best Estimator**<u>

CatBoostRegressor(**bagging_temperature=0,\
                depth=6**, \
                early_stopping_rounds=100, \
                eval_metric='Quantile:alpha=0.987531903035467', \
                iterations=1000,\
                **l2_leaf_reg=5, \
                learning_rate=0.03**, \
                loss_function='Quantile:alpha=0.987531903035467', \
                verbose=50)
                
<u>**Huber Loss Hyperparameter Best Estimator**<u>
    
CatBoostRegressor(**bagging_temperature=0,\
                depth=8**, \
                early_stopping_rounds=100, \
                eval_metric='Huber:delta=2.0', \
                iterations=1000, \
                **l2_leaf_reg=10, \
                learning_rate=0.01**, \
                loss_function='Huber:delta=2.0',\
                verbose=50)

In [84]:
# Do a train_test_split on the data, with stratification
X_train, X_test, y_train, y_test = train_test_split(df[features],df.LENGTH_OF_STAY,
                                                    shuffle = True,
                                                    random_state = 2222,
                                                    stratify = df['STRATA'],
                                                   test_size = .2)

# Create the StratifiedKFold
strata_train = df.loc[X_train.index, 'STRATA']

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [86]:
# Run the models for each metric on the train set, recording each metric.
# Dictionary defining different metrics and tuned hyperparameters [bagging_temperature, depth, l2_leaf_reg, learning_rate]
metrics = {"RMSE" : [0,6,3,0.03],
           "MAE": [0,7,5,0.05],
           "Huber:delta=2.0": [0,8,10,0.01],
           "Quantile:alpha=0.987531903035467": [0,6,5,0.03]}

scores = pd.DataFrame(
    0.0,
    index=["RMSE", "MAE", "Huber", "Quantile"],
    columns=["RMSE", "MAE", "Huber", "Quantile"]
)

row_names = {
    "RMSE": "RMSE",
    "MAE": "MAE",
    "Huber:delta=2.0": "Huber",
    "Quantile:alpha=0.987531903035467": "Quantile"
}

for metric,params in metrics.items():
    fold_rmse = []
    fold_mae = []
    fold_huber = []
    fold_quantile = []
    # Fit the model for each fold
    
    print(f"\nTraining {metric} model...")
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
        X_tr = X_train.iloc[train_idx].copy()
        X_val = X_train.iloc[valid_idx].copy()
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        # Instantiate Model
        model = CatBoostRegressor(
            bagging_temperature = params[0], #Tuned Hyperparameter 1
            depth=params[1], #Tuned Hyperparameter 2
            l2_leaf_reg=params[2], #Tuned Hyperparameter 3
            learning_rate=params[3], #Tuned Hyperparameter 4
            iterations=3000,
            loss_function=metric, # metrics looped through 4 options
            eval_metric=metric,
            random_seed=1987,
            early_stopping_rounds=100,
            verbose=100
        )

        model.fit(X_tr,
            y_tr,
            cat_features = cat_features,
            eval_set = (X_val, y_val),
            use_best_model=True)

        # Find predictions and store rmses  
        preds = model.predict(X_val)
    
        rmse_pred = rmse(y_val,preds)
        mae_pred = mae(y_val,preds)
        quantile_pred = mean_pinball_loss(
            y_val,
            preds,
            alpha=0.987531903035467
        )

        huber_pred = huber_loss(y_val, preds, delta=2.0)

        fold_rmse.append(rmse_pred)
        fold_mae.append(mae_pred)
        fold_huber.append(huber_pred)
        fold_quantile.append(quantile_pred)

        print(
        f"{metric} | Fold {fold+1}/{skf.get_n_splits()} | "
        f"RMSE={rmse_pred:.3f}, "
        f"MAE={mae_pred:.3f}, "
        f"Huber={huber_pred:.3f}, "
        f"Quantile={quantile_pred:.5f}"
        )

    row = row_names[metric]
    
    scores.loc[row, "RMSE"] = np.mean(fold_rmse)
    scores.loc[row, "MAE"] = np.mean(fold_mae)
    scores.loc[row, "Huber"] = np.mean(fold_huber)
    scores.loc[row, "Quantile"] = np.mean(fold_quantile)

    print(f"Finished {metric} model.\n")


Training RMSE model...
0:	learn: 14.1641447	test: 16.0697331	best: 16.0697331 (0)	total: 1.42s	remaining: 1h 10m 50s
100:	learn: 2.6729885	test: 4.7112081	best: 4.7112081 (100)	total: 49.6s	remaining: 23m 44s
200:	learn: 1.6336931	test: 4.1138503	best: 4.1138503 (200)	total: 1m 31s	remaining: 21m 21s
300:	learn: 1.2020205	test: 4.1474129	best: 4.1092423 (210)	total: 2m 14s	remaining: 20m 9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 4.109242303
bestIteration = 210

Shrink model to first 211 iterations.
RMSE | Fold 1/5 | RMSE=4.109, MAE=0.235, Huber=0.082, Quantile=0.11936
0:	learn: 14.8310922	test: 13.4149640	best: 13.4149640 (0)	total: 697ms	remaining: 34m 51s
100:	learn: 1.8530711	test: 1.6882825	best: 1.6882825 (100)	total: 43.1s	remaining: 20m 37s
200:	learn: 0.7655952	test: 1.0684360	best: 1.0684360 (200)	total: 1m 23s	remaining: 19m 28s
300:	learn: 0.5137382	test: 0.9611980	best: 0.9608760 (298)	total: 2m 2s	remaining: 18m 21s
400:	learn: 0.3984204	test: 

In [87]:
# See scores for each model across each metric
scores

,RMSE,MAE,Huber,Quantile
RMSE,2.360747,0.233531,0.117676,0.117935
MAE,9.021104,0.052183,0.087770,0.044646
Huber,112.865799,26.146870,51.326854,11.522051
Quantile,19.531626,0.718422,1.191194,0.016856
